In [2]:
import os
from pathlib import Path

import pandas as pd
import shioaji as sj
from dotenv import load_dotenv


In [3]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

API_KEY = os.environ["SHIOAJI_API_KEY"]
SECRET_KEY = os.environ["SHIOAJI_SECRET_KEY"]
SIMULATION = os.getenv("SHIOAJI_SIMULATION", "1") == "1"

START, END = "2026-10-01", "2026-10-02"
SYMBOL = "TXFR1"

OUT_DIR = PROJECT_ROOT / "data"
OUT_DIR.mkdir(exist_ok=True)
print("simulation =", SIMULATION, "| out =", OUT_DIR)

simulation = True | out = /Users/chihjuihsu/Documents/python_script/shioaji_api/data


In [5]:
api = sj.Shioaji(simulation=SIMULATION)
accounts = api.login(api_key=API_KEY, secret_key=SECRET_KEY)
accounts

Response Code: 0 | Event Code: 0 | Info: host '210.59.255.161:80', hostname '210.59.255.161:80' IP 210.59.255.161:80 (host 1 of 1) (host connection attempt 1 of 1) (total connection attempt 1 of 2) | Event: Session up


[FutureAccount(person_id='M122583116', broker_id='F002002', account_id='8125198', signed=true, username='許志睿')]

Response Code: 200 | Event Code: 18 | Info: #P2P/v:bcsolace01/Oryfkqqm/PYAPI/M122583116/1007/092700/653748000/175.183.91.96/> | Event: Session Property modification ok


In [6]:
contract = api.Contracts.Futures.TXF["TXFR1"]  # 等同 api.contracts.get("TXFR1")
print(contract)

Response Code: 200 | Event Code: 16 | Info: APISUB/V2/SYS/CONTRACT/TW/FUT | Event: Subscribe or Unsubscribe ok
FuturesInfo(Contract(security_type='FUT', region='TW', exchange='TAIFEX', code='TXFR1', target_code='TXFJ6'), code='TXFR1', name='臺股期貨 近月', root='TXF', delivery_month='202610', delivery_date=datetime.date(2026, 10, 21), last_trading_date=datetime.date(2026, 10, 21), begin_date=datetime.date(2026, 7, 16), underlying_kind='I', underlying_code='IX0001', multiplier=200.0, contract_size=200.0, size_unit='pt', quote_ccy='TWD', tick_basis='fixed', tick=1.0, tick_value=200.0, spec_kind='index_fut', decimal_locator=2, dynamic_banding=True, flow_group='10', reference=49968.0, limit_up=54964.0, limit_down=44972.0, update_date=datetime.date(2026, 10, 7))


In [7]:
kbars = api.kbars(contract, start=START, end=END)

df_k = pd.DataFrame({**kbars})
df_k["ts"] = pd.to_datetime(df_k["ts"])
df_k = df_k.set_index("ts").sort_index()
df_k = df_k[["Open", "High", "Low", "Close", "Volume"]]

print(df_k.shape)
df_k.head()

(2265, 5)


,Open,High,Low,Close,Volume
ts,,,,,
2026-10-01 00:00:00,48370.0,48385.0,48370.0,48385.0,21
2026-10-01 00:01:00,48384.0,48386.0,48369.0,48373.0,62
2026-10-01 00:02:00,48370.0,48383.0,48370.0,48377.0,9
2026-10-01 00:03:00,48377.0,48387.0,48377.0,48387.0,12
2026-10-01 00:04:00,48384.0,48384.0,48370.0,48375.0,10


In [8]:
api.logout()